# Notebook 05: Measurement Mechanics and Primitives V2

This notebook covers measurement mechanics, finite-shot sampling, and the Qiskit Primitives V2 execution architecture.

---

## Learning Objectives
1. Understand quantum measurement and wavefunction collapse.
2. Add measurement registers to quantum circuits.
3. Configure and execute jobs using `qiskit.primitives.StatevectorSampler`.
4. Process Primitive Unified Blocs (PUBs) and analyze shot statistics.


---
## Real-World Applications & Modern Use Cases

Finite-shot sampling via modern execution primitives is used in:
- **Cloud QPU Job Scheduling:** Submitting batched quantum algorithms to remote IBM Quantum hardware efficiently.
- **Quantum Monte Carlo Sampling:** Sampling risk distributions in financial quantitative analysis and calculating complex financial derivative pricing.


---
## Section 1: Creating a Circuit with Measurement Registers

Measurements in Qiskit collapse the quantum state into classical bits stored in a classical register.


In [1]:
from qiskit import QuantumCircuit

# Build a 2-qubit Bell circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

# Append measurement operations to all qubits
qc.measure_all()

print("Circuit with Measurement Operations:")
print(qc.draw(output='text'))


Circuit with Measurement Operations:
        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1


---
## Section 2: Initializing the SamplerV2

In Qiskit 1.x, sampling is performed using `StatevectorSampler` (SamplerV2), which accepts inputs in Primitive Unified Bloc (PUB) format.


In [2]:
from qiskit.primitives import StatevectorSampler

# Initialize the SamplerV2 primitive
sampler = StatevectorSampler()

print("StatevectorSampler initialized successfully.")


StatevectorSampler initialized successfully.


---
## Section 3: Executing a Sampling Job

We submit the circuit as a PUB tuple: `(circuit,)`, specifying the desired shot count directly in `sampler.run()`.


In [3]:
# Define shot count
total_shots = 2048

# Execute the job
job = sampler.run([qc], shots=total_shots)
result = job.result()

print(f"Job completed successfully. Total PUB results: {len(result)}")


Job completed successfully. Total PUB results: 1


---
## Section 4: Extracting and Analyzing Counts

We parse the classical register data (`meas`) from the result container.


In [4]:
# Extract bitstring counts from the first PUB result
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

print("Measured Bitstring Frequencies:")
for bitstring, count in counts.items():
    pct = (count / total_shots) * 100
    print(f"Bitstring '{bitstring}': {count} shots ({pct:.2f}%)")


Measured Bitstring Frequencies:
Bitstring '00': 1040 shots (50.78%)
Bitstring '11': 1008 shots (49.22%)


---
## Section 5: Visualizing Sampled Distributions

We plot the shot distribution using Matplotlib.


In [5]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(counts.keys(), counts.values(), color='#1f77b4', width=0.4)
plt.xlabel("Measured Bitstring (q1 q0)", fontsize=11)
plt.ylabel("Shot Counts", fontsize=11)
plt.title(f"SamplerV2 Shot Distribution ({total_shots} Total Shots)", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()


<string>:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


---
## Section 6: Batch Execution via Multiple PUBs

SamplerV2 allows batch execution of multiple circuits in a single call.


In [6]:
# Construct a second circuit: GHZ state (|000> + |111>)
ghz = QuantumCircuit(3)
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)
ghz.measure_all()

# Submit batch job with both circuits
batch_job = sampler.run([qc, ghz], shots=1000)
batch_results = batch_job.result()

print("Batch Job Results:")
print("Circuit 1 (Bell):", batch_results[0].data.meas.get_counts())
print("Circuit 2 (GHZ): ", batch_results[1].data.meas.get_counts())


Batch Job Results:
Circuit 1 (Bell): {'11': 508, '00': 492}
Circuit 2 (GHZ):  {'000': 506, '111': 494}
